# Project FORESIGHT — Baseline & Demand Forecasting Models

This notebook builds the first forecasting stage from the cleaned `analysis_ready_daily.csv`.

Models:
1. Seasonal Naive baseline
2. XGBoost
3. LightGBM

Evaluation:
- WAPE
- Forecast bias
- Chronological train/test split

**Important:** This is the initial model comparison. The final FORESIGHT evaluation must still use rolling-origin backtesting before selecting the production model.


In [ ]:
from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

BASE_DIR = Path.cwd()

# If this notebook is stored inside src/notebooks instead,
# change the line above to:
# BASE_DIR = Path.cwd().parent.parent

DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_DIR / "analysis_ready_daily.csv"

print("Input:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())


In [ ]:
# Load cleaned daily data

df = pd.read_csv(
    INPUT_FILE,
    parse_dates=["date"]
)

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("SKUs:", df["sku_id"].nunique())

display(df.head())


## 1. Convert daily demand to weekly SKU demand

The project requires weekly SKU-level forecasting. We therefore aggregate the cleaned daily data into one row per SKU per week.


In [ ]:
df["week_start"] = (
    df["date"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

weekly = (
    df.groupby(
        ["sku_id", "week_start"],
        as_index=False
    )
    .agg(
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        avg_unit_price=("avg_unit_price", "mean"),
        avg_discount_pct=("avg_discount_pct", "mean"),
        transaction_count=("transaction_count", "sum")
    )
    .sort_values(["sku_id", "week_start"])
    .reset_index(drop=True)
)

print("Weekly rows:", len(weekly))
print("SKUs:", weekly["sku_id"].nunique())

display(weekly.head())


In [ ]:
# Check weekly coverage

coverage = (
    weekly
    .groupby("sku_id")["week_start"]
    .nunique()
)

display(coverage.describe())


## 2. Complete the SKU-week grid

Missing observations after a SKU's first observed sale are treated as zero weekly demand for this initial model.

Periods before a SKU's first observed sale are kept out of the target series.


In [ ]:
all_weeks = pd.date_range(
    weekly["week_start"].min(),
    weekly["week_start"].max(),
    freq="W-MON"
)

all_skus = weekly["sku_id"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_skus, all_weeks],
    names=["sku_id", "week_start"]
)

weekly = (
    weekly
    .set_index(["sku_id", "week_start"])
    .reindex(complete_index)
    .reset_index()
)

first_sale = (
    df.groupby("sku_id")["date"]
    .min()
    .rename("first_sale_date")
)

weekly = weekly.merge(
    first_sale,
    on="sku_id",
    how="left"
)

before_first_sale = (
    weekly["week_start"] < weekly["first_sale_date"]
)

weekly.loc[before_first_sale, "units_sold"] = np.nan

after_first_sale = ~before_first_sale

weekly.loc[after_first_sale, "units_sold"] = (
    weekly.loc[after_first_sale, "units_sold"]
    .fillna(0)
)

weekly["revenue"] = weekly["revenue"].fillna(0)
weekly["avg_discount_pct"] = weekly["avg_discount_pct"].fillna(0)
weekly["transaction_count"] = weekly["transaction_count"].fillna(0)

weekly["avg_unit_price"] = (
    weekly
    .groupby("sku_id")["avg_unit_price"]
    .transform(lambda x: x.ffill().bfill())
)

weekly = weekly.drop(columns=["first_sale_date"])

weekly = weekly.sort_values(
    ["sku_id", "week_start"]
).reset_index(drop=True)

print("Complete SKU-week rows:", len(weekly))
display(weekly.head())


## 3. Calendar and historical-demand features

In [ ]:
weekly["year"] = weekly["week_start"].dt.year
weekly["month"] = weekly["week_start"].dt.month
weekly["quarter"] = weekly["week_start"].dt.quarter
weekly["week_of_year"] = (
    weekly["week_start"]
    .dt.isocalendar()
    .week
    .astype(int)
)

# Global time trend
weekly["trend"] = (
    (weekly["week_start"] - weekly["week_start"].min())
    .dt.days // 7
)


In [ ]:
# Lag features

weekly = weekly.sort_values(
    ["sku_id", "week_start"]
)

group = weekly.groupby("sku_id")

for lag in [1, 2, 4, 8, 13, 26]:
    weekly[f"lag_{lag}"] = (
        group["units_sold"].shift(lag)
    )

# Rolling features.
# shift(1) is essential: it prevents current-week demand
# from entering its own features.

group_demand = weekly.groupby("sku_id")["units_sold"]

for window in [4, 8, 13]:
    weekly[f"rolling_mean_{window}"] = (
        group_demand
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

for window in [4, 8]:
    weekly[f"rolling_std_{window}"] = (
        group_demand
        .shift(1)
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
    )

display(weekly.head(30))


## 4. Prepare model data

In [ ]:
feature_columns = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "lag_13",
    "lag_26",
    "rolling_mean_4",
    "rolling_mean_8",
    "rolling_mean_13",
    "rolling_std_4",
    "rolling_std_8",
    "avg_unit_price",
    "avg_discount_pct",
    "month",
    "quarter",
    "week_of_year",
    "trend"
]

weekly_model = weekly.dropna(
    subset=feature_columns
).copy()

# Encode SKU globally.
# The same mapping is used for train and test.

sku_categories = pd.Categorical(
    weekly_model["sku_id"]
)

weekly_model["sku_code"] = (
    sku_categories.codes
)

features = [
    "sku_code",
    *feature_columns
]

target = "units_sold"

print("Model rows:", len(weekly_model))
print("Features:", len(features))


## 5. Chronological train/test split

The final year is kept as the initial test period.

This is intentionally a time-based split rather than a random split.


In [ ]:
train_end = pd.Timestamp("2024-12-31")

train = weekly_model[
    weekly_model["week_start"] <= train_end
].copy()

test = weekly_model[
    weekly_model["week_start"] > train_end
].copy()

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

print("Training:", train["week_start"].min(), "→", train["week_start"].max())
print("Test:", test["week_start"].min(), "→", test["week_start"].max())

print("\nTrain rows:", len(train))
print("Test rows:", len(test))


## 6. Evaluation metrics

In [ ]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        return np.nan

    return np.sum(
        np.abs(y_true - y_pred)
    ) / denominator


def forecast_bias(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return np.mean(y_pred - y_true)


## 7. Seasonal-Naive baseline

In [ ]:
test_baseline = test.copy()

# Previous week's demand
test_baseline["baseline_prediction"] = (
    test_baseline["lag_1"]
)

baseline_wape = wape(
    test_baseline["units_sold"],
    test_baseline["baseline_prediction"]
)

baseline_bias = forecast_bias(
    test_baseline["units_sold"],
    test_baseline["baseline_prediction"]
)

print(f"Seasonal Naive WAPE: {baseline_wape:.4f}")
print(f"Seasonal Naive Bias: {baseline_bias:.4f}")


## 8. XGBoost

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="mae",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_pred = xgb_model.predict(X_test)
xgb_pred = np.maximum(xgb_pred, 0)

xgb_wape = wape(y_test, xgb_pred)
xgb_bias = forecast_bias(y_test, xgb_pred)

print(f"XGBoost WAPE: {xgb_wape:.4f}")
print(f"XGBoost Bias: {xgb_bias:.4f}")


## 9. LightGBM

In [ ]:
lgb_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_model.fit(
    X_train,
    y_train
)

lgb_pred = lgb_model.predict(X_test)
lgb_pred = np.maximum(lgb_pred, 0)

lgb_wape = wape(y_test, lgb_pred)
lgb_bias = forecast_bias(y_test, lgb_pred)

print(f"LightGBM WAPE: {lgb_wape:.4f}")
print(f"LightGBM Bias: {lgb_bias:.4f}")


## 10. Compare models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Seasonal Naive",
        "XGBoost",
        "LightGBM"
    ],
    "WAPE": [
        baseline_wape,
        xgb_wape,
        lgb_wape
    ],
    "Bias": [
        baseline_bias,
        xgb_bias,
        lgb_bias
    ]
}).sort_values("WAPE").reset_index(drop=True)

display(results)

best_model_name = results.iloc[0]["Model"]

print("Best model on this initial test:", best_model_name)

if best_model_name == "Seasonal Naive":
    print(
        "\nThe ML models did not beat the baseline. "
        "Do not hide this result; investigate with rolling-origin backtesting."
    )
else:
    print(
        f"\n{best_model_name} currently beats the baseline "
        "on this initial test split."
    )


## 11. Feature importance

In [ ]:
xgb_importance = pd.Series(
    xgb_model.feature_importances_,
    index=features
).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
xgb_importance.head(15).sort_values().plot(kind="barh")
plt.title("XGBoost — Top 15 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 12. Actual vs forecast for one SKU

In [ ]:
example_sku = test["sku_id"].iloc[0]

mask = test["sku_id"].values == example_sku

plot_df = test.loc[mask, [
    "week_start",
    "sku_id",
    "units_sold",
    "lag_1"
]].copy()

plot_df["baseline_prediction"] = plot_df["lag_1"].values
plot_df["xgb_prediction"] = xgb_pred[mask]
plot_df["lgb_prediction"] = lgb_pred[mask]

plt.figure(figsize=(14, 6))

plt.plot(
    plot_df["week_start"],
    plot_df["units_sold"],
    label="Actual"
)

plt.plot(
    plot_df["week_start"],
    plot_df["baseline_prediction"],
    label="Seasonal Naive"
)

plt.plot(
    plot_df["week_start"],
    plot_df["xgb_prediction"],
    label="XGBoost"
)

plt.plot(
    plot_df["week_start"],
    plot_df["lgb_prediction"],
    label="LightGBM"
)

plt.title(f"Demand Forecast — {example_sku}")
plt.xlabel("Week")
plt.ylabel("Units Sold")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 13. Save models

These are the initial models. The final production model should be selected only after the required rolling-origin backtest.


In [ ]:
joblib.dump(
    xgb_model,
    MODEL_DIR / "xgboost_demand_model.pkl"
)

joblib.dump(
    lgb_model,
    MODEL_DIR / "lightgbm_demand_model.pkl"
)

joblib.dump(
    features,
    MODEL_DIR / "model_features.pkl"
)

# Save the SKU mapping used by the models.
sku_mapping = pd.DataFrame({
    "sku_id": sku_categories.categories,
    "sku_code": range(len(sku_categories.categories))
})

sku_mapping.to_csv(
    MODEL_DIR / "sku_mapping.csv",
    index=False
)

results.to_csv(
    MODEL_DIR / "initial_model_comparison.csv",
    index=False
)

print("Models and comparison results saved to:", MODEL_DIR)


## Next stage

This notebook gives the initial baseline/model comparison.

Before calling a model the final FORESIGHT model, implement:

**Rolling-origin cross-validation → WAPE comparison → final model selection → 6–8 week forecasting.**

The project brief requires rolling-origin validation and explicitly says that a complex model should only be retained if it beats the seasonal-naive baseline honestly.
